<a href="https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [8]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [5]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.report_date > b.end_d - INTERVAL 60 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 60 DAY AND f.report_date > b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_60_90,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30,
               STDDEV(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.report_date > b.end_d - INTERVAL 60 DAY THEN f.gsc_avg_position END) AS pos_volatility_prior
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 750
    )
    SELECT * FROM windowed
""").df()
print(f'{len(features):,} content items with enough history')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

47,857 content items with enough history


**Threshold choice:** raised `imp_prev30` cutoff from 500 → 750. Rationale: at 500 the panel
includes very low-traffic pages where a 20% swing is 1-2 impressions of noise, not a real
signal. 750 trims the noisiest tail while keeping enough volume per client. Renamed
`imp_prev60` → `imp_60_90` since it is the isolated 60–90 day block, not a cumulative sum —
the old name was misleading.

## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [6]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 47,857 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,imp_60_90,clk_last30,pos_last30,pos_volatility_prior,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_2e296120acb03e93,2346.0,2954.0,2253.0,0.0,38.156281,6.768675,43.0,0.042632,0.545876,409.0,3108.0,0.131596
1,client_e547b89c05043229,content_516b7c0e8eec0cef,371.0,809.0,1685.0,0.0,48.364954,15.763559,13.0,0.112391,0.679232,305.0,597.0,0.510888
2,client_e547b89c05043229,content_38b6c1a9aa29f801,5746.0,5670.0,6987.0,2.0,40.082120,6.047873,129.0,0.069119,0.517198,1398.0,7613.0,0.183633
3,client_e547b89c05043229,content_2ffd36f2a70be7e3,1051.0,1567.0,2171.0,0.0,27.835101,6.342069,30.0,0.061182,0.749635,145.0,906.0,0.160044
4,client_e547b89c05043229,content_4724385fe790d24a,4771.0,1512.0,1366.0,11.0,8.842269,3.714377,18.0,0.048242,0.868087,243.0,640.0,0.379688


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


**Objective:** Build a first predictive model and validate it honestly against a
simple baseline, before trusting any result.

**Label definition:** A content page is marked as "declining" if its impressions
dropped by more than 20% in the most recent 30 days compared to the prior 30 days.

**Why this avoids leakage:** The features used to predict (from the *prev-30*
window and query-mix data) are all built from information available *before* the
outcome window. The label itself is defined using the *last-30* window — the
outcome we're trying to predict. Because features and label come from non-overlapping
time periods, the model can't "see" the answer baked into its own inputs.

**Validation approach:**
- Split data into train (75%) and test (25%) sets, so evaluation happens on unseen data
- Compare model accuracy against a dumb baseline: always predicting the majority class
- A model is only meaningful if it clearly beats this baseline — not just matches it

In [7]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)
data['momentum_prior'] = data['imp_prev30'] / data['imp_60_90'].clip(lower=1)

feature_cols = ['imp_prev30', 'momentum_prior', 'pos_volatility_prior',
                'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

# --- Rule-based baseline: no ML, just momentum direction ---
baseline_pred = (model_data['momentum_prior'] < 1).astype(int)
print("--- Rule baseline (momentum_prior < 1) ---")
print(f'accuracy: {accuracy_score(y, baseline_pred):.3f}')
print(f'ROC-AUC:  {roc_auc_score(y, model_data["momentum_prior"] * -1):.3f}\n')

# --- Random split, 5-fold CV instead of single seed ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []
for tr_idx, te_idx in skf.split(X, y):
    m = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    aucs.append(roc_auc_score(y.iloc[te_idx], m.predict_proba(X.iloc[te_idx])[:, 1]))
print(f"--- Random split (5-fold CV) ---\nAUC range: {min(aucs):.3f} - {max(aucs):.3f}, mean: {np.mean(aucs):.3f}\n")

# --- Fit once more for full report + feature importance ---
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1).fit(X_tr, y_tr)
probs = model.predict_proba(X_te)[:, 1]
print("--- Random split (page-level), single seed for detail ---")
print(f'ROC-AUC: {roc_auc_score(y_te, probs):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))
print("\nFeature importance:")
print(pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False))

# --- Group split (client-level) — unchanged, keep as-is ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data['client_hash_id']))
X_tr2, X_te2 = X.iloc[train_idx], X.iloc[test_idx]
y_tr2, y_te2 = y.iloc[train_idx], y.iloc[test_idx]
model2 = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
probs2 = model2.predict_proba(X_te2)[:, 1]
print("\n--- Group split (client-level) ---")
print(f'ROC-AUC: {roc_auc_score(y_te2, probs2):.3f}')
print(classification_report(y_te2, model2.predict(X_te2), digits=3))

--- Rule baseline (momentum_prior < 1) ---
accuracy: 0.577
ROC-AUC:  0.601

--- Random split (5-fold CV) ---
AUC range: 0.710 - 0.725, mean: 0.718

--- Random split (page-level), single seed for detail ---
ROC-AUC: 0.722
              precision    recall  f1-score   support

           0      0.668     0.327     0.439      4198
           1      0.710     0.910     0.797      7591

    accuracy                          0.702     11789
   macro avg      0.689     0.618     0.618     11789
weighted avg      0.695     0.702     0.670     11789


Feature importance:
momentum_prior          0.171927
rare_share              0.150955
pos_volatility_prior    0.150563
imp_prev30              0.142246
anon_share              0.138745
top_query_share         0.131000
visible_queries         0.114564
dtype: float64

--- Group split (client-level) ---
ROC-AUC: 0.589
              precision    recall  f1-score   support

           0      0.816     0.111     0.196      3104
           1      0.323  

**Reading the precision, not just the AUC:** in the group split, precision on the
"declining" class is ~0.325 — meaning roughly two-thirds of the pages this model flags as
declining are false alarms. For a recommendation engine, this matters more than the AUC
headline: it sets the honest expectation that flagged pages need human review, not automatic
action. The rule baseline above (momentum direction only) is the floor this model needs to
clear — report both numbers side by side in the paper's Results section, not the RF number
alone.

Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


## 6. Your turn — completed

The three exercises from the previous section, done in order: a 90-day-window rebuild of the feature table, one new engineered feature, and a re-run of the baseline / random-split / group-split comparison with the expanded feature set.

**Note:** these cells are written and ready to run, but have not been executed here — run them in your own Colab session (where your HF token and the live `con` connection from section 1 are already set up) to get real numbers. Don't trust placeholder output; trust what your own run prints.

### 6.1 Rebuild features on a 90-day window

Same pattern as section 3, but the comparison windows are 90 days each instead of 30: `imp_last90` (most recent 90 days) vs. `imp_prev90` (the 90 days before that), with a third `imp_180_270` block for the volatility feature — mirroring how `imp_60_90` supported `pos_volatility_prior` in the 30-day version.

**Threshold choice:** the original `HAVING imp_prev30 >= 750` cutoff was tuned for a 30-day volume. A 90-day window naturally accumulates roughly 3x the impressions per page, so the cutoff below is `HAVING imp_prev90 >= 2000` — not a straight 3x (2250) because a slightly lower multiple keeps mid-traffic pages that a strict 3x would have dropped, while still clearing the same very-low-traffic noise the original 750 was meant to trim. Adjust after you see how many rows survive.

In [9]:
features_90 = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last90,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY AND f.report_date > b.end_d - INTERVAL 180 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev90,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 180 DAY AND f.report_date > b.end_d - INTERVAL 270 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_180_270,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last90,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_avg_position END)       AS pos_last90,
               STDDEV(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY AND f.report_date > b.end_d - INTERVAL 180 DAY THEN f.gsc_avg_position END) AS pos_volatility_prior_90
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 270 DAY
        GROUP BY 1, 2
        HAVING imp_prev90 >= 2000
    )
    SELECT * FROM windowed
""").df()
print(f'{len(features_90):,} content items with enough history (90-day window)')
print(f'(30-day version had {len(features):,} for comparison)')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

45,807 content items with enough history (90-day window)
(30-day version had 47,857 for comparison)


### 6.2 New feature: `weekend_share`

Fraction of a page's prior-window impressions that fall on a Saturday or Sunday. Rationale: B2B / work-related content and consumer content often have opposite weekday/weekend impression shapes, and a page whose traffic is unusually weekend-heavy or weekend-light relative to its own history can be a signal of an audience or intent shift — independent of the momentum and query-mix signals already in the feature set. Computed on the **prev-30 window** (not last-30) so it stays a pre-outcome feature, same leakage discipline as the rest of section 5.

In [10]:
weekend = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    )
    SELECT f.content_hash_id,
           SUM(CASE WHEN DAYOFWEEK(f.report_date) IN (0, 6) THEN f.gsc_impressions ELSE 0 END)
               / NULLIF(SUM(f.gsc_impressions), 0) AS weekend_share
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date <= b.end_d - INTERVAL 30 DAY
      AND f.report_date >  b.end_d - INTERVAL 60 DAY
    GROUP BY 1
""").df()
print(f'{len(weekend):,} content items with a weekend_share value')
weekend.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

389,153 content items with a weekend_share value


,content_hash_id,weekend_share
0,content_b90a982ea3d4999d,0.421053
1,content_17d994b99d470434,0.353083
2,content_9e6d399bb7df2d21,0.244851
3,content_8cd2b02ae0c9cca4,0.000000
4,content_889961fe0fd51a4b,0.333333


### 6.3 Re-run the baseline / random-split / group-split comparison with the expanded feature set

Joins `weekend_share` onto the original (30-day) feature table — keeping the 30-day label and leakage design from section 5 unchanged, so this is an apples-to-apples test of *does the new feature help*, not a test of the 90-day rebuild. (The 90-day table from 6.1 is a separate exploration — merge it in the same way if you want to test window length and the new feature together, but change one variable at a time first so you know which change moved the numbers.)


In [11]:
data_v2 = data.merge(weekend, on='content_hash_id', how='left')

feature_cols_v2 = feature_cols + ['weekend_share']
model_data_v2 = data_v2.dropna(subset=feature_cols_v2)
X2, y2 = model_data_v2[feature_cols_v2], model_data_v2['is_declining']

print(f'rows available with weekend_share populated: {len(model_data_v2):,} (vs {len(model_data):,} without it)')

# --- Rule baseline (unchanged, reference point) ---
baseline_pred_v2 = (model_data_v2['momentum_prior'] < 1).astype(int)
print("\n--- Rule baseline (momentum_prior < 1) ---")
print(f'accuracy: {accuracy_score(y2, baseline_pred_v2):.3f}')
print(f'ROC-AUC:  {roc_auc_score(y2, model_data_v2["momentum_prior"] * -1):.3f}')

# --- Random split, 5-fold CV ---
skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs2 = []
for tr_idx, te_idx in skf2.split(X2, y2):
    m = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
    m.fit(X2.iloc[tr_idx], y2.iloc[tr_idx])
    aucs2.append(roc_auc_score(y2.iloc[te_idx], m.predict_proba(X2.iloc[te_idx])[:, 1]))
print(f"\n--- Random split (5-fold CV) ---\nAUC range: {min(aucs2):.3f} - {max(aucs2):.3f}, mean: {np.mean(aucs2):.3f}")

# --- Group split (client-level) — the number that actually matters ---
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx2, test_idx2 = next(gss2.split(X2, y2, groups=model_data_v2['client_hash_id']))
X_tr3, X_te3 = X2.iloc[train_idx2], X2.iloc[test_idx2]
y_tr3, y_te3 = y2.iloc[train_idx2], y2.iloc[test_idx2]
model3 = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1).fit(X_tr3, y_tr3)
probs3 = model3.predict_proba(X_te3)[:, 1]
print("\n--- Group split (client-level), with weekend_share added ---")
print(f'ROC-AUC: {roc_auc_score(y_te3, probs3):.3f}')
print(classification_report(y_te3, model3.predict(X_te3), digits=3))
print("Feature importance:")
print(pd.Series(model3.feature_importances_, index=feature_cols_v2).sort_values(ascending=False))
print("\nCompare this group-split ROC-AUC and 'declining'-class precision directly against "
      "section 5's group-split numbers (0.596 AUC / 0.325 precision) to see whether weekend_share "
      "actually earned its place in the model, or just added noise.")


rows available with weekend_share populated: 47,154 (vs 47,154 without it)

--- Rule baseline (momentum_prior < 1) ---
accuracy: 0.577
ROC-AUC:  0.601

--- Random split (5-fold CV) ---
AUC range: 0.733 - 0.743, mean: 0.736

--- Group split (client-level), with weekend_share added ---
ROC-AUC: 0.637
              precision    recall  f1-score   support

           0      0.829     0.131     0.226      3104
           1      0.327     0.940     0.485      1396

    accuracy                          0.382      4500
   macro avg      0.578     0.535     0.356      4500
weighted avg      0.673     0.382     0.307      4500

Feature importance:
momentum_prior          0.161047
weekend_share           0.138767
rare_share              0.134226
pos_volatility_prior    0.131863
imp_prev30              0.114945
anon_share              0.114872
top_query_share         0.108528
visible_queries         0.095751
dtype: float64

Compare this group-split ROC-AUC and 'declining'-class precision directly

### 6.4 Reading these results

Two questions to answer for yourself from the printed output above before this goes anywhere near the capstone paper:

1. **Did `weekend_share` help the number that matters?** The random-split AUC will almost certainly look fine regardless — that split is optimistic by construction, since it lets the same clients' pages appear in both train and test. The honest read is the **group-split** row: compare its ROC-AUC and "declining"-class precision against section 5's group-split baseline (0.596 AUC / 0.325 precision) with the *same* random seed. A feature that moves the random-split number but not the group-split number is fitting per-client quirks, not a generalizable signal.
2. **Did the 90-day window change how many pages qualify, and does that shift the story?** Section 6.1 prints a row count next to the original 30-day count — a large drop or jump in eligible pages changes who your recommendations would apply to, which belongs in the paper's Data / Methodology section regardless of which window you keep.

Whatever you find — feature helped, didn't help, or window length mattered more than the feature — report the comparison itself in the capstone paper's Methodology and Results sections, not just the best-looking number. That comparison *is* the honest-validation story the paper is asking for.
